In [1]:
# Pdf Extraction using PyPDF

In [2]:
!pip install pymupdf

  Obtaining dependency information for pymupdf from https://files.pythonhosted.org/packages/f5/fb/a3f1f8813f6e93c65d1f7ebca6530a889f1ae109229b537f7a617b2aab57/pymupdf-1.27.2-cp310-abi3-win_amd64.whl.metadata
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB 1.3 MB/s eta 0:00:16
   ---------------------------------------- 0.1/19.2 MB 1.1 MB/s eta 0:00:19
   ---------------------------------------- 0.2/19.2 MB 1.7 MB/s eta 0:00:12
    --------------------------------------- 0.4/19.2 MB 2.4 MB/s eta 0:00:08
   - -------------------------------------- 0.7/19.2 MB 3.0 MB/s eta 0:00:07
   -- ------------------------------------- 1.1/19.2 MB 4.0 MB/s eta 0:00:05
   -- ------------------------------------- 1.3/19.2 MB 4.1 MB/s eta 0:00:05
   --- ------------------------------------ 1.5/19.2 MB 4.3 MB/s eta 0:00:05
   --- ------------------------------------ 1.7/19.2 MB 4.6 MB/s eta 0:00:04
   ---- ---------------------

Importing Libraries

In [1]:
import fitz
import pandas as pd
import re

Extracting data from the pdf

In [2]:
pdf_path = 'datasets\\a1988-59.pdf'

doc = fitz.open(pdf_path)

text = ""
for page in doc:
    text += page.get_text()

print("Text Length: ", len(text))

Text Length:  377391


Cleaning the text by adding spaces and sub

In [3]:
text = text.replace("\n", " ")
text = re.sub(r'\s+', ' ', text)

Detecting the chapters and sections, subsections as our dataset consists of the part such as 
(no): what the section is<br>
<t>(no): subsections explaining about the terms that also false under the section 

In [4]:
rows = []

current_chapter = None
current_section = None

tokens = re.split(r'(?=\bCHAPTER\b|\d+\.)', text)

for token in tokens:
    
    # Detect chapter
    chap = re.search(r'CHAPTER\s+([IVXLC]+)', token)
    if chap:
        current_chapter = chap.group(1)
    
    # Detect section
    sec = re.match(r'(\d+)\.\s*(.*)', token)
    if sec:
        current_section = sec.group(1)
        rows.append({
            "chapter": current_chapter,
            "section": current_section,
            "subsection": "main",
            "text": sec.group(2).strip()
        })
    
    # Detect subsections (i), (ii), (iii)
    subs = re.findall(r'\((i+|v|x+)\)\s*(.*?)(?=\(\w+\)|Explanation|$)', token)
    for s in subs:
        rows.append({
            "chapter": current_chapter,
            "section": current_section,
            "subsection": s[0],
            "text": s[1].strip()
        })
    
    # Detect explanation
    exp = re.search(r'Explanation\.\—(.*)', token)
    if exp:
        rows.append({
            "chapter": current_chapter,
            "section": current_section,
            "subsection": "explanation",
            "text": exp.group(1).strip()
        })

In [5]:
df = pd.DataFrame(rows)
df.head(20)

,chapter,section,subsection,text
0,I,1,main,"Short title, extent and commencement."
1,I,2,main,Definitions. 2A. e-cart and e-rickshaw.
2,II,3,main,Necessity for driving licence.
3,II,4,main,Age limit in connection with driving of motor ...
4,II,5,main,Responsibility of owners of motor vehicles for...
5,II,4,main,
6,II,6,main,Restrictions on the holding of driving licences.
7,II,7,main,Restrictions on the granting of learner’s lice...
8,II,8,main,Grant of learner’s licence.
9,II,9,main,Grant of driving licence.


In [6]:
df.to_csv("Motor_Vehicle_Act_Dataset.csv", index= False)
print("Dataset saved successfully")

Dataset saved successfully


In [7]:
def chunk_text(text, size= 200):
    words = text.split()
    for i in range(0, len(words), size):
        yield " ".join(words[i:i+size])

In [19]:
df1 = pd.read_csv('datasets\\Motor_Vehicle_Act_Dataset.csv')
df1.head()

,chapter,section,subsection,text
0,I,1,main,"Short title, extent and commencement."
1,I,2,main,Definitions. 2A. e-cart and e-rickshaw.
2,II,3,main,Necessity for driving licence.
3,II,4,main,Age limit in connection with driving of motor ...
4,II,5,main,Responsibility of owners of motor vehicles for...
